# 工具使用
## 1.工具调用
### 1.1直接调用

In [1]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    # This is a placeholder implementation. In a real scenario, you would call a weather API.
    return city + " is sunny with a temperature of 25°C."

In [3]:
get_weather.invoke({"city": "New York"})

'New York is sunny with a temperature of 25°C.'

### 1.2 基于模型调用

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv(override=True)
# 初始化模型
model = init_chat_model(
    model="glm-4.5-air",
    model_provider="openai",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL")
)


In [10]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    # This is a placeholder implementation. In a real scenario, you would call a weather API.
    return city + "晴天，温度25°C。"

# ！绑定工具
model_with_tools = model.bind_tools([get_weather])

# AI决定是否调用工具
response = model_with_tools.invoke("北京天气如何？")
# response = model_with_tools.invoke("2+3=？")

# 检查AI是否要调用工具
if response.tool_calls:
    print("AI wants to call a tool:", response.tool_calls)
else:
    print("AI does not want to call any tools.")
print("AI response:", response.text)

AI wants to call a tool: [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_-7480176374846582226', 'type': 'tool_call'}]
AI response: 
我来帮您查询北京的天气情况。



### 1.4 从message流转看工具的调用

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os
from langchain_core.tools import tool
from rich import print as rprint

load_dotenv(override=True)
# 初始化模型
model = init_chat_model(
    model="glm-4.5-air",
    model_provider="openai",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL")
)

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    # This is a placeholder implementation. In a real scenario, you would call a weather API.
    return city + "晴天，温度25°C。"

# ！绑定工具
model_with_tools = model.bind_tools([get_weather])

messages = []
messages.append(HumanMessage(content="北京天气如何？"))
response = model_with_tools.invoke(messages)
messages.append(response)
# rprint(response)
tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)

print("="*10+"messages"+"="*10)
for msg in messages:
    msg.pretty_print()
print("="*10+"messages"+"="*10)
final_response = model_with_tools.invoke(messages)
rprint(f"final_response: {final_response}")


<class 'langchain_core.messages.tool.ToolMessage'>
==========messages==========
================================ Human Message =================================

北京天气如何？
================================== Ai Message ==================================


我来帮您查询北京的天气情况。
Tool Calls:
  get_weather (call_-7480185102220126747)
 Call ID: call_-7480185102220126747
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京晴天，温度25°C。
==========messages==========


final_response: content='\n北京今天天气很好，是晴天，温度为25°C，是个不错的天气呢！' additional_kwargs={'refusal': 
None} response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 200, 'total_tokens': 261, 
'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 199}}, 
'model_provider': 'openai', 'model_name': 'glm-4.5-air', 'system_fingerprint': None, 'id': 
'2026070215533336a1faa2b5f94550', 'finish_reason': 'stop', 'logprobs': None} 
id='lc_run--019f21d1-9f0b-7df0-8a46-07af37bf40c0-0' tool_calls=[] invalid_tool_calls=[] 
usage_metadata={'input_tokens': 200, 'output_tokens': 61, 'total_tokens': 261, 'input_token_details': 
{'cache_read': 199}, 'output_token_details': {}}